## Identificação da equipe


- **Equipe:** Grupo 2
- **Integrantes:** Eduardo Gabriel Souza Cardozo; Gabriel Alves de Farias; Gisele Franco de Lima; Luigi Santos Caires; Rodrigo de Souza Galvão
- **Turma:** Matutino 
- **Data da coleta:** 18/09/2026


#Introdução
### Coleta de Dados — RIPE Atlas

Este notebook implementa a primeira etapa do projeto de predição
de falhas de rede.

O objetivo desta etapa é realizar a coleta de dados de medições
de rede utilizando a API do RIPE Atlas.

Os dados coletados serão posteriormente tratados e utilizados
como entrada para um modelo baseado em árvore de decisão.

## Decisões da equipe e Justificativas

Cada equipe deve pesquisar e definir os próprios parâmetros. Registre abaixo:

| Decisão | Escolha da equipe | Justificativa |
|---|---|---|
| ID da medição (`measurement_id`) | 1001 | Mais Abaixo |
| Intervalo consultado | 10 Minutos | Mais Abaixo |
| Duração do intervalo | 10 Minutos | Mais Abaixo |
| Tipo de medição | Pública | Mais Abaixo |
| Formato de saída: CSV ou Parquet | CSV | Mais Abaixo |

Antes de escolher, consultem a documentação e verifiquem se a medição é pública, se o ID existe e se o intervalo solicitado possui resultados. Evitem períodos excessivamente longos: eles podem retornar muitos registros ou tornar a consulta lenta.

Justificativas.
Metrica 1001
A medição ID 1001 do RIPE Atlas é uma das medições nativas e contínuas (built-in measurements) da plataforma gerenciada pelo RIPE NCC. K-Root server (k.root-servers.net), um dos 13 servidores-raiz do sistema de nomes de domínio (DNS) oficiais da Internet, operado diretamente pelo próprio RIPE NCC.

O ID é 1001, o tipo de teste é ping(ICMP) e seu objetivo é avaliar continuamente o tempo de resposta (latência) e a conectividade global em relação ao servidor K-root a partir de milhares de sondas (probes) espalhadas pelo mundo. Por ser uma medição pública e estrutural, qualquer usuário pode consumir os dados históricos ou em tempo real dessa medição sem custos de créditos da plataforma.

# Roteiro da Implementação

1. Importação das bibliotecas
2. Configuração da pasta raw
3. Configuração da API
4. Função de coleta de dados
5. Execução da coleta
6. Visualização da resposta da API
7. Função de conversão de JSON em DataFrame
8. Visualização dos dados
9. Salvar os dados na pasta raw
10. Execução das funções de exportação dos dados
11. Contribuições individuais
12. Checklist da equipe

## 1. Importação das bibliotecas

In [89]:
import requests
import pandas as pd
import datetime as dt
import json

from pathlib import Path

print('Carregamento de bibliotecas concluído.')

Carregamento de bibliotecas concluído.


## 2. Configuração da pasta raw

Deve-se escolher o bloco a ser executado dependendo do ambiente na qual esse notebook estiver sendo executado

In [54]:
# caso esteja sendo executado no Visual Studio Code, o caminho relativo para a pasta raw será esse
# RAW_FOLDER_PATH = '../raw'
# print('Localização da pasta raw: ', RAW_FOLDER_PATH)

In [90]:
# caso esteja sendo executado no Google Colab, o caminho de pastas será esse
from google.colab import drive

drive.mount("/content/drive")

PROJECT_FOLDER = Path("/content/drive/MyDrive/ripe_atlas")
RAW_FOLDER_PATH = PROJECT_FOLDER / 'raw'
RAW_FOLDER_PATH.mkdir(parents=True, exist_ok=True)

print('Localização da pasta raw: ', RAW_FOLDER_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Localização da pasta raw:  /content/drive/MyDrive/ripe_atlas/raw


## 3. Configuração da API

In [91]:
# métricas definidas pela equipe
MEASUREMENT_ID = 1001
MEASUREMENT_MINUTES_TIME = 10
EXPORTATION_FORMAT = 'csv'
TIMEOUT = 60
PUBLIC_ONLY = True

# gera dois datetimes, sendo um deles com um tempo definido de diferença
# para retornar dados atualizados da API em um intervalo
stop_datetime = dt.datetime.today()
start_datetime = stop_datetime - dt.timedelta(minutes=MEASUREMENT_MINUTES_TIME)

API_URL = f'https://atlas.ripe.net/api/v2/measurements/{MEASUREMENT_ID}/results/'

# sempre coleta dados atualizados, levando em conta a hora atual
START_TIME = start_datetime.isoformat()
STOP_TIME = stop_datetime.isoformat()

# parametros da requisição
params = {
    'start': START_TIME,
    'stop': STOP_TIME,
    'public_only': str(PUBLIC_ONLY).lower()
}

print('Configuração da API')
print("URL:", API_URL)
print("Início (UTC):", start_datetime.isoformat())
print("Fim (UTC):", stop_datetime.isoformat())
print("Parâmetros:", params)

Configuração da API
URL: https://atlas.ripe.net/api/v2/measurements/1001/results/
Início (UTC): 2026-09-18T16:52:53.217567
Fim (UTC): 2026-09-18T17:02:53.217567
Parâmetros: {'start': '2026-09-18T16:52:53.217567', 'stop': '2026-09-18T17:02:53.217567', 'public_only': 'true'}


## 4. Função de coleta de dados

In [92]:
def collect_data(url: str, params: dict, timeout: int):
    response = requests.get(
        url,
        params=params,
        timeout=timeout
    )

    response.raise_for_status()
    return response, response.json()

## 5. Execução da coleta


In [93]:
request_start_time = dt.datetime.now()
try:
    response, raw_json_data = collect_data(API_URL, params, TIMEOUT)

except requests.exceptions.Timeout as error:
    raise RuntimeError("Tempo de requisição excedido. A API não respondeu dentro do tempo limite.") from error
except requests.exceptions.HTTPError as error:
    raise RuntimeError(f"Erro na requisição HTTP: {error.response.status_code} - {error.response.reason}") from error
except requests.exceptions.RequestException as error:
    raise RuntimeError(f"Erro na comunicação com a API: {error}") from error
except ValueError as error:
    raise RuntimeError(f"Erro ao processar a resposta da API: {error}") from error
except Exception as error:
    raise RuntimeError(f"Ocorreu um erro inesperado: {error}") from error

if not raw_json_data:
    raise RuntimeError("Nenhum dado foi retornado pela API. Verifique os parâmetros da requisição.")

assert isinstance(raw_json_data, list)

print('Código de status HTTP: ', response.status_code)
print('Quantidade de registros: ', len(raw_json_data))
print('Tipo da resposta da API: ', type(raw_json_data))

Código de status HTTP:  200
Quantidade de registros:  30780
Tipo da resposta da API:  <class 'list'>


## 6. Visualização da resposta da API

In [94]:
print('Início da requisição: ', request_start_time)

print(json.dumps(raw_json_data[0], indent=4))


Início da requisição:  2026-09-18 17:02:57.496717
{
    "fw": 5080,
    "mver": "2.6.2",
    "lts": 8,
    "dst_name": "193.0.14.129",
    "af": 4,
    "dst_addr": "193.0.14.129",
    "src_addr": "192.168.178.30",
    "proto": "ICMP",
    "ttl": 55,
    "size": 20,
    "result": [
        {
            "rtt": 15.260238
        },
        {
            "rtt": 15.563465
        },
        {
            "rtt": 16.044286
        }
    ],
    "dup": 0,
    "rcvd": 3,
    "sent": 3,
    "min": 15.260238,
    "max": 16.044286,
    "avg": 15.622663000000001,
    "msm_id": 1001,
    "prb_id": 10001,
    "timestamp": 1789750376,
    "msm_name": "Ping",
    "from": "45.138.229.91",
    "type": "ping",
    "step": 240,
    "stored_timestamp": 1789750421
}


## 7. Função para Conversão de JSON para DataFrame

In [95]:
def normalize_relevant_data(data: list) -> pd.DataFrame | None:
    if not data:
        return None

    return pd.json_normalize(data, sep='_')


## 8. Visualização dos dados

In [96]:
raw_df = normalize_relevant_data(raw_json_data)
display(raw_df.head())

,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,max,avg,msm_id,prb_id,timestamp,msm_name,from,type,step,stored_timestamp
0,5080,2.6.2,8,193.0.14.129,4,193.0.14.129,192.168.178.30,ICMP,55.0,20,...,16.044286,15.622663,1001,10001,1789750376,Ping,45.138.229.91,ping,240,1789750421
1,5130,2.6.4,22,193.0.14.129,4,193.0.14.129,192.168.77.31,ICMP,55.0,32,...,2.768479,2.638893,1001,1000222,1789750375,Ping,87.117.219.165,ping,240,1789750412
2,5120,2.6.4,7,193.0.14.129,4,193.0.14.129,169.255.0.135,ICMP,62.0,32,...,0.569302,0.461859,1001,1000492,1789750378,Ping,169.255.0.135,ping,240,1789750446
3,5020,2.2.1,10825543,193.0.14.129,4,193.0.14.129,90.156.136.55,ICMP,58.0,32,...,10.288270,9.969535,1001,1000807,1789750379,Ping,90.156.136.55,ping,240,1789750234
4,5100,2.6.4,22,193.0.14.129,4,193.0.14.129,194.58.31.15,ICMP,55.0,32,...,130.913507,121.957842,1001,1000903,1789750379,Ping,194.58.31.15,ping,240,1789750435


In [97]:
# Visualização geral do dataframe
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30780 entries, 0 to 30779
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   fw                30780 non-null  int64  
 1   mver              29791 non-null  object 
 2   lts               30780 non-null  int64  
 3   dst_name          30780 non-null  object 
 4   af                30780 non-null  int64  
 5   dst_addr          30780 non-null  object 
 6   src_addr          30780 non-null  object 
 7   proto             30780 non-null  object 
 8   ttl               29727 non-null  float64
 9   size              30780 non-null  int64  
 10  result            30780 non-null  object 
 11  dup               30780 non-null  int64  
 12  rcvd              30780 non-null  int64  
 13  sent              30780 non-null  int64  
 14  min               30780 non-null  float64
 15  max               30780 non-null  float64
 16  avg               30780 non-null  float6

## 9. Salvar os dados na pasta raw

Arquivos que devem ser salvos:

- Um arquivo .json com a resposta original, sem transformação;
- Um arquivo .csv ou .parquet com o conteúdo exibido no DataFrame;
- Um arquivo de metadados .json contendo as evidências da coleta.

In [98]:
# função para exportação do arquivo json com resposta original
def export_raw_json(
    raw_json_data: list,
    filename: str,
    path: str
) -> str:
    """
    Essa função realiza a exportação da resposta bruta da API em JSON e
    retorna uma String contendo o caminho do arquivo
    """
    if not raw_json_data:
        raise ValueError("Os dados JSON estão vazios. Não há dados para exportar.")

    filename = f'{path}/{filename}.json'

    with open(filename, 'w') as file:
        json.dump(raw_json_data, file, indent=4)

    return filename

In [99]:
# função para exportação do dataframe
def export_raw_df(
    df_raw: pd.DataFrame,
    filename: str,
    format: str,
    path: str
) -> str:
    """
    Essa função realiza a exportação de um dataframe em CSV contendo as
    métricas coletadas da API e retorna uma String com o nome do arquivo
    """
    if df_raw is None or df_raw.empty:
        raise ValueError("O DataFrame está vazio. Não há dados para exportar.")

    filename = f'{path}/{filename}.{format}'

    if format == 'csv':
        df_raw.to_csv(filename, index=False)
    else:
        raise ValueError(f"Formato de exportação '{format}' não suportado.")

    return filename

In [100]:
# função para exportação dos metadados do dataframe
def export_metadata(
        raw_json_data: list,
        raw_df: pd.DataFrame,
        measurement_id: int,
        url: str,
        params: dict,
        start_datetime: dt.datetime,
        stop_datetime: dt.datetime,
        request_start_time: dt.datetime,
        base_filename: str,
        path: str,
        exportation_format: str
) -> str:
    """
    Essa função faz a criação de um dicionário com todas os principais metadados
    relacionados à um dataframe exportado em CSV, e retorna uma String contendo
    o caminho do arquivo
    """
    metadata = {
        'measurement_id': measurement_id,
        'url': url,
        'params': params,
        'start_datetime': start_datetime.isoformat(),
        'stop_datetime': stop_datetime.isoformat(),
        'request_start_time': request_start_time.isoformat(),
        'total_records': len(raw_json_data),
        'total_columns': len(raw_df.columns),
        'json_file': f'{path}/{base_filename}.json',
        'df_file': f'{path}/{base_filename}.{exportation_format}'
    }

    metadata_filename = f'{path}/{base_filename}_metadata.json'
    with open(metadata_filename, 'w') as file:
        json.dump(metadata, file, indent=4)

    return metadata_filename

## 10. Execução das funções de exportação dos dados

In [101]:
# com base no horário de execução da requisição cria um timestamp para o nome do arquivo
file_timestamp = request_start_time.strftime("%Y%m%dT%H%M%S")
base_filename = f'ripe_atlas_m{MEASUREMENT_ID}_{file_timestamp}'

raw_json_path = export_raw_json(
    raw_json_data,
    base_filename,
    RAW_FOLDER_PATH
)

raw_df_path = export_raw_df(
    raw_df,
    base_filename,
    EXPORTATION_FORMAT,
    RAW_FOLDER_PATH
)

metadata_path = export_metadata(
    raw_json_data,
    raw_df,
    MEASUREMENT_ID,
    API_URL,
    params,
    start_datetime,
    stop_datetime,
    request_start_time,
    base_filename,
    RAW_FOLDER_PATH,
    EXPORTATION_FORMAT
)

print("Caminho do Arquivo JSON:", raw_json_path)
print("Caminho do CSV exportado:", raw_df_path)
print("Caminho dos Metadados do CSV:", metadata_path)

print('\n\nColeta realizada com sucesso')

Caminho do Arquivo JSON: /content/drive/MyDrive/ripe_atlas/raw/ripe_atlas_m1001_20260918T170257.json
Caminho do CSV exportado: /content/drive/MyDrive/ripe_atlas/raw/ripe_atlas_m1001_20260918T170257.csv
Caminho dos Metadados do CSV: /content/drive/MyDrive/ripe_atlas/raw/ripe_atlas_m1001_20260918T170257_metadata.json


Coleta realizada com sucesso


In [102]:
# exibição do arquivo de metadados
try:
    with open(metadata_path, 'r') as file:
        data = json.load(file)
        print("Metadados:", json.dumps(data, indent=4))
except FileNotFoundError:
    print("O arquivo de metadados não foi encontrado")

Metadados: {
    "measurement_id": 1001,
    "url": "https://atlas.ripe.net/api/v2/measurements/1001/results/",
    "params": {
        "start": "2026-09-18T16:52:53.217567",
        "stop": "2026-09-18T17:02:53.217567",
        "public_only": "true"
    },
    "start_datetime": "2026-09-18T16:52:53.217567",
    "stop_datetime": "2026-09-18T17:02:53.217567",
    "request_start_time": "2026-09-18T17:02:57.496717",
    "total_records": 30780,
    "total_columns": 25,
    "json_file": "/content/drive/MyDrive/ripe_atlas/raw/ripe_atlas_m1001_20260918T170257.json",
    "df_file": "/content/drive/MyDrive/ripe_atlas/raw/ripe_atlas_m1001_20260918T170257.csv"
}


## 11. Contribuições individuais

Integrante 1 — `Eduardo Gabriel de Souza Cardozo`

* **Atividade realizada:**
  `Pesquisei a parte sobre as metricas do ID 1001 e justifiquei porque a usamos.`

* **Parte do trabalho relacionada:**
  `Seção 2. Decisões da Equipe.`

* **Tempo dedicado (aproximado):**
  `1h30`

* **Evidência da contribuição:**
  `https://drive.google.com/drive/folders/13olukf8vVRO3G_QmUzqMcU0w3mnNOOSM?usp=drive_link`

* **Explicação da evidência:**
  `Comprova que eu decide e tive aprovação da minha equipe para assumir a responsabilidade por essa parte. Também mostra o resultado final.`

Integrante 2 — `[Nome completo do aluno]`

* **Atividade realizada:**
  `[Descreva exatamente o que você produziu, pesquisou, analisou, programou, testou ou revisou.]`

* **Parte do trabalho relacionada:**
  `[Informe a seção, arquivo, código, tarefa ou decisão em que você participou.]`

* **Tempo dedicado (aproximado):**
  `[Ex.: 3h30]`

* **Evidência da contribuição:**
  `[Insira o link do repositório dos arquivos que esta usando como evidência]`

* **Explicação da evidência:**
  `[Explique brevemente o que essa evidência comprova e onde sua contribuição pode ser identificada.]`

Integrante 3 — `Gisele Franco de Lima`

* **Atividade realizada:**
  `Realizei o preenchimento e a organização da documentação do projeto, incluindo a identificação da equipe, o checklist e o registro final da coleta de dados, consolidando as informações obtidas durante a execução do notebook.`

* **Parte do trabalho relacionada:**
  `Documentação da etapa de coleta de dados, registro final da equipe e organização das informações para a entrega do projeto.`

* **Tempo dedicado (aproximado):**
  `[Ex.: 2h00]`

* **Evidência da contribuição:**
  `https://drive.google.com/drive/folders/1TObWd6UN5ZlC3QF06j6ee0YCiUsn2jmh?usp=sharing`

* **Explicação da evidência:**
  `A evidência apresenta as seções do documento preenchidas por mim, incluindo a identificação da equipe, o checklist e o registro final da coleta de dados. Os registros demonstram diretamente a participação na organização e documentação das informações referentes à etapa realizada pela equipe.`


Integrante 4 — `[Nome completo do aluno]`

* **Atividade realizada:**
  `[Descreva exatamente o que você produziu, pesquisou, analisou, programou, testou ou revisou.]`

* **Parte do trabalho relacionada:**
  `[Informe a seção, arquivo, código, tarefa ou decisão em que você participou.]`

* **Tempo dedicado (aproximado):**
  `[Ex.: 3h30]`

* **Evidência da contribuição:**
  `[Insira o link do repositório dos arquivos que esta usando como evidência]`

* **Explicação da evidência:**
  `[Explique brevemente o que essa evidência comprova e onde sua contribuição pode ser identificada.]`

Integrante 5 — `Rodrigo de Souza Galvão`

* **Atividade realizada:**
  `Programei o notebook de coleta de dados, fiz a implementação de funções para exportação dos dados nos formatos exigidos: JSON, CSV, metadados e realizei testes para garantir que tudo está funcionando.`

* **Parte do trabalho relacionada:**
  `Arquivo do notebook: ripe_atlas_data_collection.ipynb`

* **Tempo dedicado (aproximado):**
  `4h20`

* **Evidência da contribuição:**
  `https://drive.google.com/drive/folders/1FB-YkgTjkZC7czWl30wIuHmJGXy_avMP?usp=sharing`

* **Explicação da evidência:**
  `Uma print demonstra eu compartilhando no grupo atualizações sobre o projeto e uma etapa concluída enquanto eu estava desenvolvendo o projeto. A outra print demonstra o histórico de commits, condizente com o arquivo que eu estava atualizando.`




**Observação**:

Evidências aceitas
1. histórico de edição de documento compartilhado;
2. commit ou pull request no GitHub;
3. arquivo ou trecho de código produzido;
4. relatório, planilha, diagrama, apresentação ou rascunho;
5. registro de tarefa no Trello, GitHub Projects ou ferramenta semelhante;
6. registro de testes realizados;
7. print de reunião, conversa ou e-mail relacionado à atividade;
8. outro material que demonstre claramente a contribuição individual.

Sendo que a evidência deverá identificar o aluno e estar relacionada à atividade declarada. Sempre que possível, apresente materiais com data, autoria ou histórico de edição. Verifique se todos os links estão acessíveis. Trabalhos realizados em conjunto também devem indicar a contribuição específica de cada integrante. Respostas genéricas, como “ajudei no trabalho”, “fiz a pesquisa” ou “participei do código”, não serão consideradas suficientes. Prints de conversas podem complementar a comprovação, mas não devem ser a única evidência quando houver um produto verificável, como código, documento, planilha ou apresentação.
A ausência de descrição ou de evidência poderá impedir a validação da contribuição individual.

## 12. Checklist da equipe

* [x] Identificamos os integrantes e a turma;
* [x] Justificamos o ID da medição e o intervalo escolhido;
* [x] A requisição terminou com status HTTP 200;
* [x] O JSON retornou pelo menos um registro;
* [x] Exibimos o `DataFrame` no notebook;
* [x] Salvamos o JSON original na pasta `raw`;
* [x] Salvamos o DataFrame em CSV na pasta `raw`;
* [x] Mantivemos o arquivo de metadados da coleta;
* [x] Não realizamos limpeza, agregação, rotulação ou treinamento nesta etapa.

### Registro final da equipe

A equipe realizou a coleta de dados utilizando a medição `1001` do RIPE Atlas, do tipo Ping/ICMP, considerando um intervalo de 10 minutos. A requisição foi realizada com `public_only` habilitado e apresentou status HTTP 200, retornando 30.780 registros organizados em 25 colunas. Os dados foram convertidos para um DataFrame e armazenados na pasta `raw` em três arquivos: JSON com a resposta original, CSV com os dados estruturados e JSON contendo os metadados da coleta. Nesta etapa, não foram realizadas limpeza, agregação, rotulação ou treinamento dos dados.

